# 02 Dataset profiling

## Purpose
Characterise the two selected recordings before any detection content is written:
what telemetry types exist, how many records each contributes, how the event-time
fields behave, and which fields are reliable enough to build a rule on. A technique
assessment built on one telemetry type understates the data, so this notebook exists
to make the second type visible.

## Inputs
- `data/processed/DS001.jsonl` and `data/processed/DS002.jsonl`, produced by
  `scripts/normalize_dataset.py`.
- `docs/DS001-normalization.json` and `docs/DS002-normalization.json`, the
  acceptance reports for those files.
- `docs/DS001-inventory.json`, `docs/DS002-inventory.json`, the provenance records.

## Methodology
The digest of each processed file is checked against its normalization report
before a single record is read. If either differs the notebook raises rather than
profiling altered data. Records are then grouped by provider and event ID, and
event-time fields are summarised per telemetry type so the timestamp policy can be
checked against the data rather than assumed.

## Outputs
A printed profile, and the durable artefact `docs/DS001-fields.json` and
`docs/DS002-fields.json`, which the compliance audit reads to confirm the telemetry
type count.

## Conclusion
Written at the end of the notebook, after the figures above are produced.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name != "Cyberion-ThreatShield":
    ROOT = Path(r"{root}")
VENV = (ROOT / ".venv/Scripts/python.exe").resolve()
if Path(sys.executable).resolve() != VENV:
    raise RuntimeError(
        "Select the Cyberion (.venv) kernel. This analysis must run on the pinned "
        f"environment, not {sys.executable}."
    )
sys.path.insert(0, str(ROOT / "scripts"))

import hashlib
import json

import pandas as pd

In [ ]:
def load_processed():
    """Load normalized records, refusing to proceed on an unverified file."""
    frames = {}
    for dataset in ("DS001", "DS002"):
        report_path = ROOT / ("docs/" + dataset + "-normalization.json")
        report = json.loads(report_path.read_text(encoding="utf-8"))
        if report["status"] != "PASS":
            raise RuntimeError(dataset + " normalization did not pass")
        path = ROOT / ("data/processed/" + dataset + ".jsonl")
        digest = hashlib.sha256(path.read_bytes()).hexdigest().upper()
        if digest != report["output_sha256"].upper():
            raise RuntimeError(dataset + " processed file digest differs from the report")
        frames[dataset] = pd.read_json(path, lines=True, convert_dates=False)
    return frames

In [ ]:
frames = load_processed()
for dataset, frame in frames.items():
    print(f"=== {dataset}: {len(frame)} records")
    providers = frame["provider"].value_counts()
    for provider, count in providers.items():
        top = frame[frame["provider"] == provider]["event_id"].value_counts().head(6)
        print(f"  {provider:<38} {count:>5}  top events {dict(top)}")

In [ ]:
print("Event-time availability by provider")
for dataset, frame in frames.items():
    print(f"--- {dataset}")
    for provider, group in frame.groupby("provider"):
        resolved = int(group["timestamp_utc"].notna().sum())
        print(f"  {provider:<38} resolved {resolved:>5} / {len(group):<5} "
              f"({resolved / len(group) * 100:5.1f}%)")

In [ ]:
# Which fields are dense enough to build a rule on? A field that is present in
# fewer than half its provider's records cannot carry a reliable condition.
for dataset, frame in frames.items():
    sysmon = frame[(frame["provider"] == "Microsoft-Windows-Sysmon")]
    print(f"--- {dataset} Sysmon field density, n={len(sysmon)}")
    for field in ("image", "command_line", "parent_image", "process_guid",
                  "parent_process_guid", "destination_ip", "script_block_text"):
        if field not in sysmon:
            continue
        density = sysmon[field].notna().sum() / len(sysmon) * 100
        print(f"  {field:<22} {density:5.1f}%")

In [ ]:
# Durable artefact: the field profile the compliance audit consumes.
for dataset, frame in frames.items():
    profile = {}
    for field in sorted({key for record in frame["raw"] for key in record}):
        counts = frame["raw"].map(lambda record: record.get(field) is not None).sum()
        if counts:
            profile[field] = int(counts)
    output = {
        "dataset_id": dataset,
        "records": int(len(frame)),
        "providers": {k: int(v) for k, v in frame["provider"].value_counts().items()},
        "channels": {k: int(v) for k, v in frame["channel"].value_counts().items()},
        "raw_field_presence": dict(sorted(profile.items(), key=lambda item: -item[1])),
    }
    (ROOT / f"docs/{dataset}-fields.json").write_text(
        json.dumps(output, indent=2) + "\n", encoding="utf-8", newline="\n")
    print(f"{dataset}: wrote docs/{dataset}-fields.json with "
          f"{len(output['raw_field_presence'])} fields")

## Conclusion

Three telemetry types carry the records: Sysmon, Windows Security auditing and
PowerShell Operational. Sysmon dominates both recordings and is the only source of
resolved event time, which is why the timestamp policy admits nothing else.

The field-density check is the practically important result. `image` and
`parent_image` are dense enough to anchor a rule; several correlation-friendly
fields such as `process_guid` are present on a small fraction of records, because
they only exist on the event types that carry them. A rule written against a sparse
field would silently match almost nothing, so density was measured before any rule
was drafted rather than after one failed.

The artefacts written here are the same ones the coverage and compliance tooling
read, so a change in the data would change the compliance result rather than
leaving stale documents behind.